# Ejercicio aplicado_Machine Learning con Spark

En este notebook entrenaremos un modelo de clasificacion binaria capaz de predecir la enfermedad cardiaca en base a diferentes mediciones de parametros bioquimicos. Para ello utilizaremos el dataset de Kaggle

### Importar librerias 

In [21]:
import numpy as np 
import pandas as pd 

### Importar libreria e inicializar sesion de spark

In [2]:
import findspark
findspark.init()

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('UCI Heart disease').getOrCreate()

24/11/18 11:38:26 WARN Utils: Your hostname, Gretels-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.8.31 instead (on interface en0)
24/11/18 11:38:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/18 11:38:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/11/18 11:38:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Importar de las librerias especificas de spark machine learning para utilizar la regresion logistica

In [4]:
from pyspark.ml.classification import LogisticRegression

### Del CSV leer los datos que hay en el archivo y definir que hay encabezado y que busque el schema

In [5]:
heart = spark.read.csv('data/heart.csv', 
                       inferSchema = True, 
                       header = True)

### Mostrar 5 renglones con encabezado 

In [6]:
heart.show(5)

+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
| 63|  1|  3|     145| 233|  1|      0|    150|    0|    2.3|    0|  0|   1|     1|
| 37|  1|  2|     130| 250|  0|      1|    187|    0|    3.5|    0|  0|   2|     1|
| 41|  0|  1|     130| 204|  0|      0|    172|    0|    1.4|    2|  0|   2|     1|
| 56|  1|  1|     120| 236|  0|      1|    178|    0|    0.8|    2|  0|   2|     1|
| 57|  0|  0|     120| 354|  0|      1|    163|    1|    0.6|    2|  0|   2|     1|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+
only showing top 5 rows



### Convertir el data frame a un data frame de pandas 

In [7]:
heart.toPandas().head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


### Imprimir el schema que tiene el data frame para saber que tipo de datos tiene 

In [8]:
heart.printSchema()

root
 |-- age: integer (nullable = true)
 |-- sex: integer (nullable = true)
 |-- cp: integer (nullable = true)
 |-- trestbps: integer (nullable = true)
 |-- chol: integer (nullable = true)
 |-- fbs: integer (nullable = true)
 |-- restecg: integer (nullable = true)
 |-- thalach: integer (nullable = true)
 |-- exang: integer (nullable = true)
 |-- oldpeak: double (nullable = true)
 |-- slope: integer (nullable = true)
 |-- ca: integer (nullable = true)
 |-- thal: integer (nullable = true)
 |-- target: integer (nullable = true)



### Preprocesamiento de datos

### Para poder procesar los datos para usar la regresion logistica se necesita importar de las librerias de pyspark Vectors y VectorAssembler

In [9]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

### Para confirmar el nombre de las columnas se usa .columns

In [10]:
heart.columns

['age',
 'sex',
 'cp',
 'trestbps',
 'chol',
 'fbs',
 'restecg',
 'thalach',
 'exang',
 'oldpeak',
 'slope',
 'ca',
 'thal',
 'target']

### Con VectorAssembler se le da de input las columnas que tiene el dataframe y el output sera solo una columna que se llamara features

In [11]:
assembler = VectorAssembler(
                            inputCols=['age',
                            'sex',
                            'cp',
                            'trestbps',
                            'chol',
                            'fbs',
                            'restecg',
                            'thalach',
                            'exang',
                            'oldpeak',
                            'slope',
                            'ca',
                            'thal'],
                            outputCol="features")

### Se necesita trasformar el assembler y se define como el output

In [12]:
output = assembler.transform(heart)

### mostrar los primeros 5 renglones

In [13]:
output.show(5)

+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+--------------------+
|age|sex| cp|trestbps|chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|            features|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+--------------------+
| 63|  1|  3|     145| 233|  1|      0|    150|    0|    2.3|    0|  0|   1|     1|[63.0,1.0,3.0,145...|
| 37|  1|  2|     130| 250|  0|      1|    187|    0|    3.5|    0|  0|   2|     1|[37.0,1.0,2.0,130...|
| 41|  0|  1|     130| 204|  0|      0|    172|    0|    1.4|    2|  0|   2|     1|[41.0,0.0,1.0,130...|
| 56|  1|  1|     120| 236|  0|      1|    178|    0|    0.8|    2|  0|   2|     1|[56.0,1.0,1.0,120...|
| 57|  0|  0|     120| 354|  0|      1|    163|    1|    0.6|    2|  0|   2|     1|[57.0,0.0,0.0,120...|
+---+---+---+--------+----+---+-------+-------+-----+-------+-----+---+----+------+--------------------+
only showing top 5 rows



24/11/18 11:38:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### Selecciona las columnas llamadas "features" y "target" del DataFrame output, el target es lo que se busca predecir.

In [14]:
final_data = output.select("features",'target')

### Entrenamiento del modelo

### Para entrenar el modelo es necesario definir que cantidad de dato se va a usar para entrenar y que cantidad se va a usar para probar. Para esto se utiliza .randomSplit una vez que ya se haya vesctorizado y seleccionado lo que se busca predecir.

In [15]:
train, test = final_data.randomSplit([0.7,0.3])

### Ya que se procesaron los datos ya se puede usar la funcion de regresion logistica, se le definen los parametros.

In [16]:
lr = LogisticRegression(labelCol="target",
                        featuresCol="features")

### se usa la funcion .fit para entrenar el modelo, se ajustan los parámetros internos del modelo (lr) con base en los datos de entrada (train).


In [17]:
model = lr.fit(train)

24/11/18 11:38:31 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


### Al modelo que ya se preparo se le aplica nuevamente el metodo de transform y se selecciona solo  las columnas "target" (el valor real del objetivo) y "prediction" (el valor predicho por el modelo) del DataFrame predict_test y se pide que solo se muestren los primeros 10 renglones

In [ ]:
predict_train=model.transform(train)
predict_test=model.transform(test)
predict_test.select("target","prediction").show(10)

+------+----------+
|target|prediction|
+------+----------+
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     1|       0.0|
|     1|       1.0|
|     1|       1.0|
|     1|       1.0|
|     0|       0.0|
|     1|       1.0|
|     1|       1.0|
+------+----------+
only showing top 10 rows



### Evaluación del modelo

### Se importa de la libreria BinaryClassificationEvaluator y se utiliza para medir si la prediccion fue certera o se acerco a los resultados esperados.

In [19]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(rawPredictionCol='rawPrediction',
                                          labelCol='target')

predict_test.select("target","rawPrediction","prediction","probability").show(5)


+------+--------------------+----------+--------------------+
|target|       rawPrediction|prediction|         probability|
+------+--------------------+----------+--------------------+
|     1|[-1.2071159694923...|       1.0|[0.23021174467986...|
|     1|[-0.1620474285622...|       1.0|[0.45957656200108...|
|     1|[-4.1407587408996...|       1.0|[0.01566158678807...|
|     1|[0.03489918014580...|       0.0|[0.50872390961193...|
|     1|[-0.6677874200691...|       1.0|[0.33899245135769...|
+------+--------------------+----------+--------------------+
only showing top 5 rows



### imprimir resultados de train y test

In [20]:
print("The area under ROC for train set is {}".format(evaluator.evaluate(predict_train)))

print("The area under ROC for test set is {}".format(evaluator.evaluate(predict_test)))

The area under ROC for train set is 0.9297222811108758
The area under ROC for test set is 0.8682116914562433
